# Bacpipe Tutorial

This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---

### Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Loading Audio Files](#2-loading-audio-files)
3. [Ground Truth Labels](#3-ground-truth-labels)
4. [End-to-End: bacpipe.play](#4-end-to-end-bacpipeplay)
5. [Full Pipeline — Single Model](#5-full-pipeline--single-model)
6. [Full Pipeline — Multiple Models](#6-full-pipeline--multiple-models)
7. [Generating Embeddings — Full Directory (Save to Disk)](#7-generating-embeddings--full-directory-save-to-disk)
8. [Generating Embeddings — Single File (In-Memory)](#8-generating-embeddings--single-file-in-memory)
9. [High-Level Workflow — generate_embeddings](#9-high-level-workflow--generate_embeddings)
10. [Benchmarking Classifier Performance](#10-benchmarking-classifier-performance)

---
## 0. Imports & Working Directory

Set the working directory to the repo root and import the display utility.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os

# Replace this with the path to the bacpipe repo root on your machine
os.chdir('../..')

---
## 1. Setup & Configuration

Import `bacpipe` and inspect the current configuration and settings. You can also list all available API endpoints, supported models, and the embedding dimensions for each model.

In [ ]:
import bacpipe

print('Config:')
display(bacpipe.config)

print('Settings:')
display(bacpipe.settings)

In [ ]:
# All available bacpipe API endpoints
bacpipe.__all__

In [ ]:
# All supported models
bacpipe.supported_models

In [ ]:
# Embedding dimensions per model
bacpipe.EMBEDDING_DIMENSIONS

---
## 2. Loading Audio Files

Retrieve all audio files from a directory as a list of strings. 

In [ ]:
audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_type='str'
)
audio_files

### 2.1 Extract datetime information from filenames

You can also extract datetime information from filenames — useful for datasets where recording time is encoded in the filename.

In [ ]:
# Extract datetime information from audio filenames
dt_files = [bacpipe.get_dt_filename(file) for file in audio_files]
dt_files

---
## 3. Ground Truth Labels

Load multi-label ground truth annotations and align them to the model's timestamps. Each row in the resulting array corresponds to the same time window as the model's predictions, making it ready for evaluation.

You can also generate a set of default labels for a model/dataset combination, which is useful when no annotations are available.

In [ ]:
# Load multi-label ground truth aligned to BirdNET's 3-second time bins
gt = bacpipe.ground_truth_by_model(
    'birdnet',
    audio_dir='bacpipe/tests/test_data',
    annotations_filename='annotations.csv',
    single_label=False
)
gt

In [ ]:
# Generate default labels for a model/dataset combination
dl = bacpipe.create_default_labels(
    'bacpipe/tests/test_data',
    'birdnet'
)
dl

---
## 4. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point. It runs the complete pipeline — embeddings, classification, dimensionality reduction, evaluation, and an interactive dashboard — in a single call. Ideal for a first exploration of a new dataset across multiple models. It does not return anything, but it saves all intermediate outputs to disk and launches the dashboard automatically (unless disabled in the `config.yaml` file).

In [ ]:
bacpipe.play(
    models=['birdnet', 'perch_bird', 'naturebeats'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

---
## 5. Full Pipeline — Single Model

`run_pipeline_for_single_model` runs the complete bacpipe pipeline for one model: embedding generation, classifier inference, optional dimensionality reduction, and visualisation. It returns a `Loader` object with all results accessible. It also saves all intermediate outputs to disk. It does not compute any evaluations and does not launch the dashboard. It's intended to be integrated into existing pipelines to return embeddings and predictions for further processing.

In [ ]:
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)
loader_obj

---
## 6. Full Pipeline — Multiple Models

`run_pipeline_for_models` does the same as `run_pipeline_for_single_model`, but runs the full pipeline across a list of models in one call. It returns a dictionary of `Loader` objects keyed by model name. This makes it straightforward to compare embeddings and predictions across models on the same dataset.

In [ ]:
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'naturebeats'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

display(loader_dictionary['birdnet'].metadata_dict)
display(loader_dictionary['naturebeats'].embeddings())

---
## 7. Generating Embeddings — Full Directory (Save to Disk)

Instead of using the all-in-one pipeline functions, you can also call the `Loader` and `Embedder` classes directly for more control. The `Loader` and `Embedder` classes are the core building blocks of bacpipe. You can instantiate them directly to generate embeddings for a single file without saving anything to disk. This is useful for quick experimentation or when you want to work with embeddings directly in memory.

To process all audio files in a directory and persist the results, pass `use_folder_structure=True` to the `Loader`. Bacpipe will create a timestamped output directory, run inference using multithreading, and save embeddings, metadata, and classifier predictions. Subsequent runs will detect the saved files and skip recomputation automatically. All of these steps are compbined in the functions `run_pipeline_for_single_model` and `run_pipeline_for_models` described above.

The loader_obj returned by the `Loader` are the same as those returned by `run_pipeline_for_single_model` and `run_pipeline_for_models`, so you can use them in the same way to access embeddings, metadata, and predictions.

In [ ]:
MODEL_NAME = 'birdnet'

loader_obj = bacpipe.Loader(
    'bacpipe/tests/test_data',
    MODEL_NAME,
    use_folder_structure=True
)
embed_obj = bacpipe.Embedder(MODEL_NAME, loader_obj)

# Process all files using multithreading
embed_obj.run_inference_pipeline_using_multithreading(
    
)

print('Embeddings (array):')
display(loader_obj.embeddings(return_type='array'))

print('Metadata dict:')
display(loader_obj.metadata_dict)

print('Predictions (array):')
display(loader_obj.predictions(return_type='array'))

print('Predictions (dataframe):')
display(loader_obj.predictions(return_type='dataframe'))

---
## 8. Generating Embeddings — Single File (In-Memory)

If you just want to generate embeddings for a single file without saving anything to disk, you can instantiate the `Loader` and `Embedder` classes directly without the keyword argument `use_folder_structure`. This will run the embedding generation in-memory and return the results as Python objects without saving any files. This is useful for quick experimentation or when you want to work with embeddings directly in memory.



In [ ]:
# Create a Loader without folder structure — nothing is written to disk
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings for the first audio file
embeds = embed_obj.get_embeddings_from_model(loader_obj.files[0])

print('Since embeddings were not saved, .embeddings() returns empty:')
display(loader_obj.embeddings())

print('The embeddings are still accessible via the declared variable:')
display(embeds)

print('Classifier predictions are accessible through the embedder object:')
display(embed_obj.classifier.predictions)

---
## 9. High-Level Workflow — `generate_embeddings`

`bacpipe.generate_embeddings` wraps the `Loader`/`Embedder` pattern into a single convenient call which can still be used without saving anything to disk. 

In [ ]:
# 
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    use_folder_structure=True
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings(return_type='array'))

### 6.1 Load and concatenate audio files yourself, then pass the result to bacpipe

You can also load and preprocess audio yourself and pass it directly to an `Embedder` — useful when you need custom loading logic or want to process audio that isn't stored on disk.

In [ ]:
# Alternatively: load and concatenate audio yourself, then pass it directly to an Embedder
import librosa as lb
import numpy as np

audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_type='str'
)

audio = []
for file in audio_files:
    aud, sr = lb.load(file)
    audio.extend(aud)
audio = np.array(audio)

# check if the model exists, if not, download it
bacpipe.ensure_models_exist(bacpipe.settings.model_base_path, ['naturebeats'])
# embed
embed_obj = bacpipe.Embedder('naturebeats')
embeds = embed_obj.embeddings_using_multithreading(audio)
embeds

---
## 10. Benchmarking Classifier Performance

`bacpipe.benchmark` evaluates a model's pretrained classifier against ground truth annotations. It aligns predictions and ground truth to the same timestamps, handles label mismatches with a regex fallback for hyphen/spacing differences, and returns a per-species `sklearn` classification report with precision, recall, and F1.

If predictions have already been generated for this dataset, the function loads them from disk and runs very quickly.

In [ ]:
results = bacpipe.benchmark(
    'birdnet',
    'bacpipe/tests/test_data',
    annotations_file='annotations.csv'
)
display(results)